In [1]:
import torch
from torch import nn
from torch.nn import CrossEntropyLoss
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, dataloader
from torchvision.transforms import ToTensor

In [2]:
training_data = datasets.MNIST(
        root='./data',
        train=True,
        transform=ToTensor(),
        download=True
)

testing_data = datasets.MNIST(
        root='./data',
        train=True,
        transform=ToTensor(),
        download=True
)

In [3]:
torch.__version__

'2.10.0+cu126'

In [4]:
batch_size = 64
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(testing_data, batch_size=batch_size)

for X, y in train_dataloader:
        print(f"Shape of X[N, C, H , W] = {X.shape}")
        print(f"Shape of y: {y.shape} {y.dtype}")
        break

Shape of X[N, C, H , W] = torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [5]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        pred = model(X)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(X)
            print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")


In [6]:
def reset_weights(m):
    if hasattr(m, 'reset_parameters'):
        m.reset_parameters()

In [7]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


In [8]:
device = torch.accelerator.current_accelerator().type

class NeuralNetwork(nn.Module):
        def __init__(self):
            super().__init__()
            self.flatten = nn.Flatten()
            self.linear_relu_stack = nn.Sequential(
                nn.Linear(28 * 28, 512),
                nn.ReLU(),
                nn.Linear(512, 256),
                nn.ReLU(),
                nn.Linear(256, 10),
            )

        def forward(self, x):
            x = self.flatten(x)
            logits = self.linear_relu_stack(x)
            return logits

model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=10, bias=True)
  )
)


In [9]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [10]:
model.apply(reset_weights)
epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)

Epoch 1
------------------------
loss: 2.294561 [    0/60000]
loss: 0.293105 [ 6400/60000]
loss: 0.210947 [12800/60000]
loss: 0.240483 [19200/60000]
loss: 0.143933 [25600/60000]
loss: 0.319921 [32000/60000]
loss: 0.129835 [38400/60000]
loss: 0.299711 [44800/60000]
loss: 0.281380 [51200/60000]
loss: 0.151617 [57600/60000]
Test Error: 
 Accuracy: 95.9%, Avg loss: 0.132886 

Epoch 2
------------------------
loss: 0.069016 [    0/60000]
loss: 0.096288 [ 6400/60000]
loss: 0.120491 [12800/60000]
loss: 0.094066 [19200/60000]
loss: 0.065085 [25600/60000]
loss: 0.105706 [32000/60000]
loss: 0.037229 [38400/60000]
loss: 0.165341 [44800/60000]
loss: 0.105708 [51200/60000]
loss: 0.105258 [57600/60000]
Test Error: 
 Accuracy: 96.8%, Avg loss: 0.098094 

Epoch 3
------------------------
loss: 0.038740 [    0/60000]
loss: 0.080179 [ 6400/60000]
loss: 0.074316 [12800/60000]
loss: 0.089221 [19200/60000]
loss: 0.054510 [25600/60000]
loss: 0.082516 [32000/60000]
loss: 0.022131 [38400/60000]
loss: 0.090271

In [11]:

device = torch.accelerator.current_accelerator().type
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=10, kernel_size=5, stride=1, padding=0),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.ReLU(),
            nn.Conv2d(in_channels=10, out_channels=20, kernel_size=5, stride=1, padding=0),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.ReLU()
        )
        self.linear_layers = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.3),
            nn.Linear(320, 128),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.linear_layers(x)
        return x

model2 = CNN().to(device)
print(model2)

CNN(
  (conv_layers): Sequential(
    (0): Conv2d(1, 10, kernel_size=(5, 5), stride=(1, 1))
    (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (2): ReLU()
    (3): Conv2d(10, 20, kernel_size=(5, 5), stride=(1, 1))
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): ReLU()
  )
  (linear_layers): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.3, inplace=False)
    (2): Linear(in_features=320, out_features=128, bias=True)
    (3): ReLU()
    (4): Dropout(p=0.3, inplace=False)
    (5): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [12]:
optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.001)

In [13]:
model2.apply(reset_weights)
epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n------------------------")
    train(train_dataloader, model2, loss_fn, optimizer2)
    test(test_dataloader, model2, loss_fn)


Epoch 1
------------------------
loss: 2.302203 [    0/60000]
loss: 0.422686 [ 6400/60000]
loss: 0.241317 [12800/60000]
loss: 0.339029 [19200/60000]
loss: 0.226406 [25600/60000]
loss: 0.176720 [32000/60000]
loss: 0.138325 [38400/60000]
loss: 0.159027 [44800/60000]
loss: 0.319449 [51200/60000]
loss: 0.196162 [57600/60000]
Test Error: 
 Accuracy: 97.2%, Avg loss: 0.090022 

Epoch 2
------------------------
loss: 0.079761 [    0/60000]
loss: 0.089498 [ 6400/60000]
loss: 0.131588 [12800/60000]
loss: 0.198255 [19200/60000]
loss: 0.062700 [25600/60000]
loss: 0.158592 [32000/60000]
loss: 0.091291 [38400/60000]
loss: 0.177210 [44800/60000]
loss: 0.214658 [51200/60000]
loss: 0.151203 [57600/60000]
Test Error: 
 Accuracy: 98.1%, Avg loss: 0.063324 

Epoch 3
------------------------
loss: 0.021575 [    0/60000]
loss: 0.096536 [ 6400/60000]
loss: 0.048279 [12800/60000]
loss: 0.067248 [19200/60000]
loss: 0.148053 [25600/60000]
loss: 0.083888 [32000/60000]
loss: 0.056315 [38400/60000]
loss: 0.043373